In [ ]:
######################################## CREATE AND READ JSON HANDLER ########################################

# Import and create the handler
from json_handler import JSONHandler

# # Case 0: DEFAULT inside the folder
# handler = JSONHandler("./losses_log.json")

handler = JSONHandler("train_dataset_evaluation.json")

# # Case 1: DYNAMIC vox
# handler = JSONHandler(
#     "/home/michele/code/michele_mmdet3d/data/TrainingSafeCopies/Training_BigVoxels_BigDataset_Dynamic/losses_log_BigVoxels_BigDataset_Dynamic.json")

# # Case 2: HARD vox
# handler = JSONHandler(
#     "/home/michele/code/michele_mmdet3d/data/TrainingSafeCopies/Training_BigVoxels_BigDataset_Hard/losses_log_BigVoxels_BigDataset_Hard.json")

# # Case 3: CONE-SHAPED --> Big Cones
# handler = JSONHandler(
#     "/home/michele/code/michele_mmdet3d/data/TrainingSafeCopies/ConeShaped_BigCone/losses_log.json")

# # Case 4: CONE-AUGMENTED
# handler = JSONHandler(
#     "/home/michele/code/michele_mmdet3d/data/TrainingSafeCopies/ConeAugmented/losses_log.json")

# # Case 5: MVX-Net with STD Convolution
# handler = JSONHandler(
#     "/home/michele/code/michele_mmdet3d/data/TrainingSafeCopies/MVX_SmallVoxels_StdConvolution/losses_log_MVX_SmallVoxels_StdConvolution.json")

# Read the file
dictionary_list = handler.read_json_file()
print(f"The file has the following number of elements:\n\t{len(dictionary_list)}")

In [ ]:
######################################## PROCESS THE RAW DATA ########################################

# Create a list where to store all the dictionaries for the final plotting
final_dict_list = []
# Create two temporary variables where to store the losses of the training
temp_loss = 0
temp_counter = 0
# For cycle for the processing of data
for i in range(len(dictionary_list)):
    # When the element is of type "training", update the temporary variables (data is cumulative)
    if dictionary_list[i]['type'] == "training":
        # Else, just update the temporary variables
        temp_loss += dictionary_list[i]['total_loss']
        temp_counter += 1
    # When the element is "validation", finalize the storage of training data and store the validation data
    else:
        final_dict_list.append(
            {'type': "training",
            'loss': (temp_loss/temp_counter)}
        )
        temp_loss=0
        temp_counter=0
        final_dict_list.append(
            {'type': "validation",
             'loss': dictionary_list[i]['total_loss'],
             'ap40': dictionary_list[i]['ap40'],
             'ap40_reduced_array': dictionary_list[i]['ap40_reduced_array'],
             'reduced_x_limit_array': dictionary_list[i]['reduced_x_limit_array'],
             'ap40_iou_thr_list': dictionary_list[i]['ap40_iou_thr_list'],
             'precisions_list': dictionary_list[i]['precisions_list'],
             'recalls_list': dictionary_list[i]['recalls_list'],
             'precisions_reduced_array': dictionary_list[i]['precisions_reduced_array'],
             'recalls_reduced_array': dictionary_list[i]['recalls_reduced_array']
             }
        )



for element in final_dict_list:
    print(element)

In [ ]:
# Establish the reduced_x_limit values
reduced_x_limit_array = final_dict_list[1]['reduced_x_limit_array']
print("\nList of the possible ranges (right), with the associated index (left)")
for i, element in enumerate(reduced_x_limit_array):
    print(f"{i}:\t{element}")

# Choose which reduced values to choose
list_of_reductions_to_show = [
    0,
    9,
    10, 
    11,
]

In [ ]:
######################################## PLOT THE LOSSES AND METRICS ########################################

import numpy
from plotters import *

# Define the validation interval
val_interval = 1

# AP40 multiplier ----------------> Needed if want to visualize AP40 in [0:100%] rather than [0:1]
ap40_multiplier = 100
loss_multiplier = 10

# Separate the data into lists
training_losses = [entry['loss']*loss_multiplier for entry in final_dict_list if entry['type'] == 'training']
validation_losses = [entry['loss']*loss_multiplier for entry in final_dict_list if entry['type'] == 'validation']
metric_ap40 = [entry['ap40']*ap40_multiplier for entry in final_dict_list if entry['type'] == 'validation']
# Special computations for the reduced AP40
metric_ap40_reduced_array = []
for entry in final_dict_list:
    if entry['type'] == 'validation':
        final_list_of_lists = []
        for sublist in entry['ap40_reduced_array']:
            final_list_of_lists.append(sublist * ap40_multiplier)
        metric_ap40_reduced_array.append(final_list_of_lists)

# Create a range of indices for the X axis
epochs = numpy.linspace(val_interval, (len(training_losses))*val_interval, len(training_losses))

plot_losses_metrics(epochs, training_losses, validation_losses, metric_ap40, metric_ap40_reduced_array, reduced_x_limit_array, 
                    list_of_reductions_shown=list_of_reductions_to_show, 
                    top_y_lim=100)

In [ ]:
# Select the epoch for which the PR-curve must be visualized
pr_curve_epoch = 58

# Compute the corresponding index
i = int(pr_curve_epoch/val_interval)*2-1
# Get the corresponding data
precisions = final_dict_list[i]['precisions_list']
recalls = final_dict_list[i]['recalls_list']
precisions_reduced = final_dict_list[i]['precisions_reduced_array'][0]
recalls_reduced = final_dict_list[i]['recalls_reduced_array'][0]
iou_thr_list = final_dict_list[i]['ap40_iou_thr_list']

# Plot the PR-curve
plot_precision_recall_curve(pr_curve_epoch, precisions, recalls, precisions_reduced, recalls_reduced, iou_thr_list)